In [1]:
!pip install datasets


  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached pandas-2.3.3-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached multiprocess-0.70.18-py311-none-any.whl.metadata (7.5 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata (2.4 kB)
  Using cached certifi-2025.11.12-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached hf_xet-1.2.0-cp37-abi3-win_amd64.whl.metadata (5.0 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using ca

In [ ]:
!pip install llama-cpp-python --upgrade --force-reinstall --prefer-binary
!pip install huggingface_hub


In [ ]:
!pip install transformers torch


In [2]:
from datasets import load_dataset

ds = load_dataset("NLP-FBK/dyspnea-crf-train")

c:\Users\kocak\Desktop\CFR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd
df = ds["en"].to_pandas()
df.head()

,document_id,clinical_note,annotations
0,1017490_en,"TRIAGE: \nReports having fallen accidentally, ...","[{'ground_truth': 'unknown', 'item': 'chronic ..."
1,1584581_en,"75-year-old patient, hypertensive on therapy, ...","[{'ground_truth': 'unknown', 'item': 'chronic ..."
2,1614319_en,Patient is a nursing home resident.\nPatient a...,"[{'ground_truth': 'unknown', 'item': 'chronic ..."
3,1823273_en,Admitted to the Emergency Department for pain ...,"[{'ground_truth': 'unknown', 'item': 'chronic ..."
4,661135_en,"72-year-old male, family history, former smoke...","[{'ground_truth': 'unknown', 'item': 'chronic ..."


In [4]:
features = [pair["item"] for pair in df["annotations"][0]]
print(features)
print(f"total features: {len(features)}")

for i in range(df.shape[0]):
  annotated_count = [pair["ground_truth"] != 'unknown'  for pair in df["annotations"][i]]
  print(sum(annotated_count))

['chronic pulmonary disease', 'chronic respiratory failure', 'chronic cardiac failure', 'chronic renal failure', 'chronic metabolic failure', 'chronic rheumatologic disease', 'active neoplasia', 'chronic dialysis', "duration of the patient's consciousness recovery", "duration of the patient's unconsciousness", 'first episod of epilepsy', 'known history of epilepsy', 'history of allergy', 'history of recent trauma', 'pregnancy', 'history of drug abuse', 'history of alcohol abuse', 'anticoagulants or antiplatelet drug therapy', 'presence of prodromal symptoms', 'compliance with antiepileptic therapy', 'tloc during effort', 'tloc while supine', 'antiepileptic therapy already in place', 'drowsiness, confusion, disorientation as postcritical state', 'stiffness during the episode', 'drooling during the episode', 'tonic-clonic seizures', 'poly-pharmacological therapy', 'pale skin during the episode', 'eye deviation during the episode', 'diffuse vascular disease', 'neuropsychiatric disorders',

In [7]:
report1 = df["clinical_note"][0]
print(report1)

labels = df["annotations"][0]
print(labels)

TRIAGE: 
Reports having fallen accidentally, impacting:
- occipital region with no loss of consciousness or concussion
- right trochanter 
- right knee
- right ankle
CS 15_Cincinnati negative_fluent speech_isochoric, isocyclic, and photoreactive pupils. No rigidity.
Conscious, lucid, oriented, not agitated.
wearing rigid cervical collar
BP 120/80 mm/hg, SpO2 99% on room air, RF 14 apm, HR 80 bpm r

MEDICAL ASSESSMENT: 
reports accidental fall to the ground at home following loss of support on right leg (spontaneous femur fracture?). Impact to the right side of the body, as described in triage. Currently experiencing pain in the right femur and right elbow. Head trauma without loss of consciousness. No vomiting after the episode. 
No other details at the moment

Past Medical History: 
- Arterial hypertension
- CAD (reports quadruple BPAC)
- Diabetes mellitus type 2
- Left hip prosthesis

Therapy: metformin 500 mg, iperten 1/2 tablet, enalapril 5 mg, omnic 0.4 mg, seloken 100 mg 1/2 tabl

# STEP 1: Extract Medical Entities

In [9]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import os
os.environ["TRANSFORMERS_NO_CHAT_TEMPLATES"] = "1"

# Model name
MODEL_NAME = "blaze999/Medical-NER"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,trust_remote_code = False)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME,trust_remote_code = False)

# Create NER pipeline
ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"  # groups subword tokens into full entities
)


Device set to use cpu


In [10]:
# Example medical note
note = report1

# Run NER
entities = ner_pipeline(note)

# Pretty print results
for ent in entities:
    print({
        "text": ent["word"],
        "label": ent["entity_group"],
        "confidence": round(ent["score"], 3),
        "start": ent["start"],
        "end": ent["end"]
    })

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'text': 'fallen', 'label': 'THERAPEUTIC_PROCEDURE', 'confidence': 0.107, 'start': 23, 'end': 30}
{'text': 'occipital region', 'label': 'BIOLOGICAL_STRUCTURE', 'confidence': 0.964, 'start': 57, 'end': 74}
{'text': 'loss of consciousness', 'label': 'SIGN_SYMPTOM', 'confidence': 0.947, 'start': 82, 'end': 104}
{'text': 'concussion', 'label': 'DISEASE_DISORDER', 'confidence': 0.774, 'start': 107, 'end': 118}
{'text': 'right trochanter', 'label': 'BIOLOGICAL_STRUCTURE', 'confidence': 0.974, 'start': 120, 'end': 137}
{'text': 'right knee', 'label': 'BIOLOGICAL_STRUCTURE', 'confidence': 0.93, 'start': 140, 'end': 151}
{'text': 'right ankle', 'label': 'BIOLOGICAL_STRUCTURE', 'confidence': 0.923, 'start': 153, 'end': 165}
{'text': 'CS 15_Cincinnati', 'label': 'DETAILED_DESCRIPTION', 'confidence': 0.805, 'start': 165, 'end': 182}
{'text': 'negative_fluent speech_', 'label': 'DETAILED_DESCRIPTION', 'confidence': 0.572, 'start': 182, 'end': 206}
{'text': 'isochoric', 'label': 'SIGN_SYMPTOM', 'con

# llm 

In [14]:
def load_medical_model():
    """
    Downloads and loads the Medical Qwen3 model for medical entity extraction.
    
    Returns:
        Llama: The loaded Llama model instance
        str: Path to the downloaded model file
    """
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama
    
    # Define the model ID and the specific GGUF filename
    model_id = "mradermacher/MedicalQwen3-Reasoning-14B-IT-i1-GGUF"
    model_filename = "MedicalQwen3-Reasoning-14B-IT.i1-Q6_K.gguf"
    
    # Download the GGUF file
    print(f"Downloading {model_filename} from {model_id}...")
    model_path = hf_hub_download(repo_id=model_id, filename=model_filename)
    print(f"Model downloaded to: {model_path}")
    
    # Load the LlamaCpp model with increased context window
    print("Loading LlamaCpp model...")
    llm = Llama(model_path=model_path, n_gpu_layers=0, n_ctx=4096, verbose=False)
    print("Model loaded.")
    
    return llm, model_path

# Load the model once at the beginning
llm, model_path = load_medical_model()

Model downloaded to: C:\Users\kocak\.cache\huggingface\hub\models--mradermacher--MedicalQwen3-Reasoning-14B-IT-i1-GGUF\snapshots\562d8aa8d3e32cc0945f12a598f43a8c6a259332\MedicalQwen3-Reasoning-14B-IT.i1-Q6_K.gguf
Loading LlamaCpp model...


llama_context: n_ctx_per_seq (4096) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


Model loaded.


In [15]:
# Model is already loaded in the previous cell using the load_medical_model() function
# The model 'llm' and 'model_path' variables are available for use
print(f"Model ready for use. Loaded from: {model_path}")

Model ready for use. Loaded from: C:\Users\kocak\.cache\huggingface\hub\models--mradermacher--MedicalQwen3-Reasoning-14B-IT-i1-GGUF\snapshots\562d8aa8d3e32cc0945f12a598f43a8c6a259332\MedicalQwen3-Reasoning-14B-IT.i1-Q6_K.gguf


In [18]:
# Function to bracket entities with their labels
def bracket_entities_with_labels(text, entities):
    """
    Adds brackets around entities with their labels.
    
    Args:
        text (str): Original clinical note text
        entities (list): List of NER entities with start, end, and label information
    
    Returns:
        str: Text with entities bracketed with their labels
    """
    # Sort entities by start position in reverse order to avoid index shifting
    sorted_entities = sorted(entities, key=lambda x: x['start'], reverse=True)
    
    bracketed_text = text
    for entity in sorted_entities:
        start = entity['start']
        end = entity['end']
        label = entity['entity_group']
        entity_text = text[start:end]
        
        # Create opening and closing brackets
        opening_bracket = f"[{label}]"
        closing_bracket = f"[/{label}]"
        
        # Insert brackets
        bracketed_text = bracketed_text[:end] + closing_bracket + bracketed_text[end:]
        bracketed_text = bracketed_text[:start] + opening_bracket + bracketed_text[start:]
    
    return bracketed_text

# Apply bracketing to the first clinical note
print("Original entities from NER:")
for ent in entities:
    print(f"  - {ent['word']} ({ent['entity_group']}) at [{ent['start']}:{ent['end']}]")

print("\n" + "="*50)
print("Applying bracketing to clinical note...")
bracketed_note = bracket_entities_with_labels(report1, entities)

print("\nBracketed Clinical Note:")
print(bracketed_note)

Original entities from NER:
  - fallen (THERAPEUTIC_PROCEDURE) at [23:30]
  - occipital region (BIOLOGICAL_STRUCTURE) at [57:74]
  - loss of consciousness (SIGN_SYMPTOM) at [82:104]
  - concussion (DISEASE_DISORDER) at [107:118]
  - right trochanter (BIOLOGICAL_STRUCTURE) at [120:137]
  - right knee (BIOLOGICAL_STRUCTURE) at [140:151]
  - right ankle (BIOLOGICAL_STRUCTURE) at [153:165]
  - CS 15_Cincinnati (DETAILED_DESCRIPTION) at [165:182]
  - negative_fluent speech_ (DETAILED_DESCRIPTION) at [182:206]
  - isochoric (SIGN_SYMPTOM) at [206:215]
  - is (LAB_VALUE) at [216:219]
  - ocyclic (SIGN_SYMPTOM) at [219:226]
  - photoreactive pupils (SIGN_SYMPTOM) at [231:252]
  - rigidity (SIGN_SYMPTOM) at [256:265]
  - Conscious (SIGN_SYMPTOM) at [266:276]
  - lucid (SIGN_SYMPTOM) at [277:283]
  - oriented (SIGN_SYMPTOM) at [284:293]
  - agitated (SIGN_SYMPTOM) at [298:307]
  - rigid (DETAILED_DESCRIPTION) at [316:322]
  - cervical collar (THERAPEUTIC_PROCEDURE) at [322:338]
  - BP (DIAGNOSTI

In [ ]:
# Function to extract relation triplets using LLM
def extract_relation_triplets_with_llm(llm, bracketed_text, ground_truth_annotations, system_prompt=None):
    """
    Uses LLM to extract evidence-backed relation triplets from bracketed clinical text.
    
    Args:
        llm: The loaded LLM model instance
        bracketed_text (str): Clinical note with entities bracketed
        ground_truth_annotations (list): Ground truth annotations with items and their status
        system_prompt (str): Optional custom system prompt
    
    Returns:
        dict: Mapping from each feature (entity) to its evidence-backed relation triplets
        dict: Full model output for debugging
    """
    
    # Default system prompt for evidence-backed relation extraction (per-feature)
    if system_prompt is None:
        system_prompt = """You are a medical knowledge graph extraction expert. Your task is to analyze clinical notes and extract relation triplets that are explicitly grounded in the text.

Based on the provided bracketed clinical note and ground truth annotations, extract EVIDENCE-BACKED relation triplets FOR EACH FEATURE in the format:

{
    "feature1": [
        {
            "triplet": (head_entity, relation, tail_entity),
            "evidence": "exact text span from the note that supports this triplet",
            "reasoning": "brief explanation of why this triplet justifies the feature annotation"
        },
        ...
    ],
    "feature2": [
        {
            "triplet": (head_entity, relation, tail_entity),
            "evidence": "exact text span from the note that supports this triplet",
            "reasoning": "brief explanation of why this triplet justifies the feature annotation"
        },
        ...
    ],
    ...
}

CRITICAL RULES:
1. ONLY use entities that appear in the BRACKETED TEXT - do NOT invent entities
2. Each triplet MUST be supported by an exact quote from the text
3. The reasoning must explain HOW this triplet relates to the feature's annotation status
4. If no direct evidence exists for a feature, explain WHY it was annotated based on context
5. Focus on explaining WHY features were marked as they were in ground truth annotations

Common medical relationships:
- "has_symptom" (e.g., "patient has_symptom fever")
- "diagnosed_with" (e.g., "patient diagnosed_with hypertension")
- "treated_with" (e.g., "patient treated_with antihypertensive therapy")
- "exhibits" (e.g., "patient exhibits consciousness")
- "has_history" (e.g., "patient has_history epilepsy")
- "has_condition" (e.g., "patient has_condition cardiovascular diseases")
- "shows_sign" (e.g., "patient shows_sign pale skin")
- "underwent" (e.g., "patient underwent CPR")
- "has_vital_sign" (e.g., "patient has_vital_sign spo2")
- "has_assessment" (e.g., "patient has_assessment normotensive")

Example output format:
{
    "antihypertensive therapy": [
        {
            "triplet": ("patient", "treated_with", "antihypertensive therapy"),
            "evidence": "patient is on antihypertensive therapy",
            "reasoning": "Direct mention of antihypertensive therapy justifies 'y' annotation"
        }
    ],
    "consciousness": [
        {
            "triplet": ("patient", "exhibits", "consciousness"),
            "evidence": "patient is conscious and lucid",
            "reasoning": "Explicit mention of consciousness supports positive annotation"
        }
    ]
}"""

    # Prepare ground truth context
    annotated_items = []
    for annotation in ground_truth_annotations:
        if annotation['ground_truth'] != 'unknown':
            annotated_items.append(f"- {annotation['item']}: {annotation['ground_truth']}")
    
    ground_truth_context = "\n".join(annotated_items) if annotated_items else "No specific ground truth annotations available."
    
    # Construct user prompt
    user_prompt = f"""Please extract EVIDENCE-BACKED relation triplets for EACH FEATURE from the following clinical data:

BRACKETED CLINICAL NOTE:
{bracketed_text}

GROUND TRUTH ANNOTATIONS (items with known status):
{ground_truth_context}

CRITICAL: For each feature, explain WHY it was annotated by finding supporting evidence in the text. If a feature like 'chronic metabolic failure' is marked as present but not explicitly mentioned, explain what evidence led to that conclusion.

Return a Python dictionary where each key is a feature name and the value is a list of dictionaries containing:
- triplet: (head, relation, tail)
- evidence: exact quote from text
- reasoning: how this justifies the annotation"""

    # Construct the full chat prompt using Qwen's chat template format
    prompt_template = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt}<|im_end|>\n<|im_start|>assistant\n"
    
    print("Extracting evidence-backed relation triplets with LLM...")
    print(f"Prompt length: {len(prompt_template)} characters")
    print(f"Ground truth annotations: {len([a for a in ground_truth_annotations if a['ground_truth'] != 'unknown'])} items")
    
    # Generate response
    output = llm(
        prompt_template,
        max_tokens=2048,
        temperature=0.1,
        top_p=0.9,
        repeat_penalty=1.1,
        stop=["<|im_end|>", "\n\n\n"],
        echo=False
    )
    
    print("Full model output:")
    print(output)
    
    # Extract and parse the generated text
    extracted_triplets = {}
    if "choices" in output and len(output["choices"]) > 0:
        generated_text = output["choices"][0]["text"].strip()
        print("\n--- Extracted Relation Triplets (Raw Output) ---")
        print(f"Generated text: '{generated_text}'")
        
        if generated_text:
            try:
                # Parse as Python dictionary
                parsed_triplets = eval(generated_text)
                if isinstance(parsed_triplets, dict):
                    extracted_triplets = parsed_triplets
                    print("\n--- Parsed Evidence-Backed Relation Triplets ---")
                    for feature, items in extracted_triplets.items():
                        print(f"\nFeature: {feature}")
                        for i, item in enumerate(items, 1):
                            if isinstance(item, dict) and "triplet" in item:
                                print(f"  {i}. {item['triplet']}")
                                print(f"     Evidence: {item.get('evidence', 'N/A')}")
                                print(f"     Reasoning: {item.get('reasoning', 'N/A')}")
                            else:
                                print(f"  {i}. {item}")
                else:
                    print(f"\nOutput is not a dictionary: {type(parsed_triplets)}")
                    extracted_triplets = fallback_evidence_triplet_extraction(generated_text)
            except Exception as e:
                print(f"\nError parsing model output: {e}")
                print("Trying to extract triplets manually...")
                extracted_triplets = fallback_evidence_triplet_extraction(generated_text)
    
    return extracted_triplets, output

def fallback_evidence_triplet_extraction(generated_text):
    """
    Fallback method to extract evidence-backed relation triplets when model output is not valid Python.
    
    Args:
        generated_text (str): The raw text output from the model
    
    Returns:
        dict: Dictionary mapping features to their evidence-backed relation triplets
    """
    feature_triplets = {}
    current_feature = None
    
    lines = generated_text.strip().split('\n')
    
    for line in lines:
        line = line.strip()
        
        # Try to detect feature names
        if line and not line.startswith('(') and not line.startswith('[') and ':' in line:
            feature_name = line.split(':')[0].strip().strip('"\'')
            if feature_name:
                current_feature = feature_name
                feature_triplets[current_feature] = []
                continue
        
        # Try to extract triplet-like patterns
        if line and '(' in line and ')' in line and ',' in line:
            if line.startswith('(') and line.endswith(')'):
                try:
                    triplet = eval(line)
                    if isinstance(triplet, tuple) and len(triplet) == 3:
                        evidence_item = {
                            "triplet": triplet,
                            "evidence": "Manual extraction - evidence not found",
                            "reasoning": "Manual extraction - reasoning not captured"
                        }
                        if current_feature and current_feature in feature_triplets:
                            feature_triplets[current_feature].append(evidence_item)
                        else:
                            if "general" not in feature_triplets:
                                feature_triplets["general"] = []
                            feature_triplets["general"].append(evidence_item)
                except:
                    pass
    
    return feature_triplets

print("Evidence-backed relation triplet extraction functions defined.")

Relation triplet extraction functions defined.


In [ ]:
# Load the medical LLM model for relation extraction
print("Loading medical LLM for relation extraction...")
medical_llm, _model_path = load_medical_model()

# Extract evidence-backed relation triplets using the bracketed note and ground truth annotations
print("\n" + "="*60)
print("EVIDENCE-BACKED RELATION TRIPLET EXTRACTION")
print("="*60)

ground_truth_annotations = df["annotations"][0]
relation_triplets_by_feature, full_output = extract_relation_triplets_with_llm(
    medical_llm,
    bracketed_note,
    ground_truth_annotations,
)

print("\n" + "="*60)
print("FINAL EVIDENCE-BACKED RELATION TRIPLETS FOR KNOWLEDGE GRAPH")
print("="*60)

if relation_triplets_by_feature:
    total_triplets = sum(len(items) for items in relation_triplets_by_feature.values())
    print(f"Successfully extracted {total_triplets} evidence-backed relation entries across {len(relation_triplets_by_feature)} features:\n")
    for feature, items in relation_triplets_by_feature.items():
        print(f"\nFeature: {feature}")
        for i, item in enumerate(items, 1):
            if isinstance(item, dict) and "triplet" in item:
                head, relation, tail = item["triplet"]
                evidence = item.get("evidence", "N/A")
                reasoning = item.get("reasoning", "N/A")
                print(f"  {i:2d}. ({head}) -[{relation}]-> ({tail})")
                print(f"      Evidence: {evidence}")
                print(f"      Reasoning: {reasoning}")
            else:
                # Fallback for non-dict items
                print(f"  {i:2d}. {item}")
else:
    print("No evidence-backed relation triplets were successfully extracted.")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Original clinical note length: {len(report1)} characters")
print(f"NER entities found: {len(entities)}")
print(f"Bracketed note length: {len(bracketed_note)} characters")
print(f"Ground truth annotations: {len(ground_truth_annotations)} items")
print(f"Annotated items (non-unknown): {len([a for a in ground_truth_annotations if a['ground_truth'] != 'unknown'])}")
print(f"Features with evidence-backed triplets: {len(relation_triplets_by_feature) if relation_triplets_by_feature else 0}")
print(f"Total evidence-backed entries: {sum(len(items) for items in relation_triplets_by_feature.values()) if relation_triplets_by_feature else 0}")

Loading medical LLM for relation extraction...
Model downloaded to: C:\Users\kocak\.cache\huggingface\hub\models--mradermacher--MedicalQwen3-Reasoning-14B-IT-i1-GGUF\snapshots\562d8aa8d3e32cc0945f12a598f43a8c6a259332\MedicalQwen3-Reasoning-14B-IT.i1-Q6_K.gguf
Loading LlamaCpp model...


llama_context: n_ctx_per_seq (4096) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


Model loaded.

RELATION TRIPLET EXTRACTION
Extracting relation triplets with LLM...
Prompt length: 6404 characters
Ground truth annotations: 12 items
Full model output:
{'id': 'cmpl-8b47afe2-f9f0-46a8-a7fd-1332f673fa4a', 'object': 'text_completion', 'created': 1766839259, 'model': 'C:\\Users\\kocak\\.cache\\huggingface\\hub\\models--mradermacher--MedicalQwen3-Reasoning-14B-IT-i1-GGUF\\snapshots\\562d8aa8d3e32cc0945f12a598f43a8c6a259332\\MedicalQwen3-Reasoning-14B-IT.i1-Q6_K.gguf', 'choices': [{'text': '<think>\n\n</think>\n\n{\n    "chronic metabolic failure": [\n        ("patient", "has_condition", "chronic metabolic failure")\n    ],\n    "history of allergy": [\n        ("patient", "has_history", "none known")\n    ],\n    "poly-pharmacological therapy": [\n        ("patient", "treated_with", "metformin"),\n        ("patient", "treated_with", "iperten"),\n        ("patient", "treated_with", "enalapril"),\n        ("patient", "treated_with", "omnic"),\n        ("patient", "treated_wi